FinBERT: Financial Sentiment and Embedding Pipeline

This notebook implements a Financial PhraseBank → FinBERT pipeline for financial sentiment classification and financial-text embedding extraction.

# Step 1 - import neccssary libraries

In [ ]:

import os
import random
import json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix

)

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 2 - Set random seeds for reproductibility

In [ ]:

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Step 3 - Load the dataset

In [ ]:

from pathlib import Path
import pandas as pd

# Find Sentences_AllAgree.txt automatically
filename = "Sentences_AllAgree.txt"

search_locations = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent
]

DATA_PATH = None

for location in search_locations:
    matches = list(location.rglob(filename))
    if matches:
        DATA_PATH = matches[0]
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Sentences_AllAgree.txt was not found. "
        "Please check that the file exists inside the project."
    )

print("Dataset found at:")
print(DATA_PATH)

# Load dataset
records = []

with open(DATA_PATH, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        # Financial PhraseBank format:
        # sentence@sentiment
        parts = line.rsplit("@", 1)

        if len(parts) != 2:
            continue

        sentence = parts[0].strip()
        sentiment = parts[1].strip().lower()

        records.append({
            "sentence": sentence,
            "sentiment": sentiment
        })

# Create DataFrame
df = pd.DataFrame(records)

print("\nDataset shape:", df.shape)

display(df.head())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Dataset found at:
c:\Users\ASUS\OneDrive\Desktop\Quantformer-project\QuantFormer-Multimodal-Order-Book-Sentiment-Transformer\datasets\FinancialPhraseBank-v1.0\Sentences_AllAgree.txt

Dataset shape: (2264, 2)


,sentence,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive



Sentiment distribution:
sentiment
neutral     1391
positive     570
negative     303
Name: count, dtype: int64


Load Financial PhraseBank

The Financial PhraseBank dataset is loaded from `Sentences_AllAgree.txt` and prepared for financial sentiment analysis.

# Step 4 - Split the Dataset

In [ ]:

df.shape
df.head()
df["sentiment"].value_counts()

sentiment
neutral     1391
positive     570
negative     303
Name: count, dtype: int64

Dataset Validation

The dataset is checked for missing values, duplicate sentences, sentiment classes, and class distribution.

# Step 5 — Dataset Validation

In [ ]:


print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate sentences:", df["sentence"].duplicated().sum())

print("\nSentiment classes:")
print(df["sentiment"].unique())

print("\nClass distribution:")
print(df["sentiment"].value_counts())

Number of rows: 2264
Number of columns: 2

Missing values:
sentence     0
sentiment    0
dtype: int64

Duplicate sentences: 5

Sentiment classes:
<StringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str

Class distribution:
sentiment
neutral     1391
positive     570
negative     303
Name: count, dtype: int64


 Label Encoding

The sentiment labels are converted into numerical labels for model processing.

# Step 6 — Label Encoding

In [ ]:


label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

df["label"] = df["sentiment"].map(label2id)

print("Label mapping:")
print(label2id)

print("\nEncoded dataset:")
display(df.head())

print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

Label mapping:
{'negative': 0, 'neutral': 1, 'positive': 2}

Encoded dataset:


,sentence,sentiment,label
0,"According to Gran , the company has no plans t...",neutral,1
1,"For the last quarter of 2010 , Componenta 's n...",positive,2
2,"In the third quarter of 2010 , net sales incre...",positive,2
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive,2
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive,2



Label distribution:
label
0     303
1    1391
2     570
Name: count, dtype: int64


 Train, Validation and Test Split

The dataset is divided into stratified training, validation, and test sets while maintaining the class distribution.

# Step 7 - train/validation/test Split 

In [ ]:


from sklearn.model_selection import train_test_split

#70 % Train ,15 % Validation, 15 % Test 

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['label'])

val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())


Train shape: (1584, 3)
Validation shape: (340, 3)
Test shape: (340, 3)

Train class distribution:
label
0    212
1    973
2    399
Name: count, dtype: int64

Validation class distribution:
label
0     45
1    209
2     86
Name: count, dtype: int64

Test class distribution:
label
0     46
1    209
2     85
Name: count, dtype: int64


# Step 8 - Load  Finbert Tokenizer 

In [ ]:

from transformers import AutoTokenizer 

MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("FinBERT tokenizer loaded succesfully.")
print("Model: ", MODEL_NAME)

FinBERT tokenizer loaded succesfully.
Model:  ProsusAI/finbert


# Check token lengths before choosing MAX_LENGTH
## Step 9 -- Tokenization


In [ ]:


train_token_lengths = [
len(tokenizer.encode(text, add_special_tokens=True))
    for text in train_df["sentence"]
]

print("Maximum token length:", max(train_token_lengths))
print("Average token length:", np.mean(train_token_lengths))
print("95th percentile:", np.percentile(train_token_lengths, 95))
print("99th percentile:", np.percentile(train_token_lengths, 99))



Maximum token length: 150
Average token length: 30.35669191919192
95th percentile: 58.0
99th percentile: 69.0


# Step 9 — Tokenization

In [ ]:


MAX_LENGTH = 64

def tokenize_dataframe(dataframe):
    return tokenizer(
        dataframe["sentence"].tolist(),
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True
    )

train_encodings = tokenize_dataframe(train_df)
val_encodings = tokenize_dataframe(val_df)
test_encodings = tokenize_dataframe(test_df)

print("Tokenization completed successfully.")
print("Train input shape:", np.array(train_encodings["input_ids"]).shape)
print("Validation input shape:", np.array(val_encodings["input_ids"]).shape)
print("Test input shape:", np.array(test_encodings["input_ids"]).shape)

Tokenization completed successfully.
Train input shape: (1584, 64)
Validation input shape: (340, 64)
Test input shape: (340, 64)


# Step 10 — PyTorch Dataset

In [ ]:


from torch.utils.data import Dataset

class FinancialNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item


train_dataset = FinancialNewsDataset(
    train_encodings,
    train_df["label"].tolist()
)

val_dataset = FinancialNewsDataset(
    val_encodings,
    val_df["label"].tolist()
)

test_dataset = FinancialNewsDataset(
    test_encodings,
    test_df["label"].tolist()
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

print("\nSample:")
print(train_dataset[0])

Train dataset size: 1584
Validation dataset size: 340
Test dataset size: 340

Sample:
{'input_ids': tensor([  101,  1996, 23060,  2100,  8584,  8187, 23060, 19198,  2003,  1037,
         2691,  5080,  2000,  4638,  5776,  2668,  1011,  7722,  2938, 18924,
         2504,  1998,  8187,  3446,  1012,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# Step 11 — DataLoaders

In [ ]:


from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders created successfully.")
print("Batch size:", BATCH_SIZE)
print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

DataLoaders created successfully.
Batch size: 16
Number of training batches: 99
Number of validation batches: 22
Number of test batches: 22


# Check one batch

In [ ]:


batch = next(iter(train_loader))

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

Input IDs shape: torch.Size([16, 64])
Attention mask shape: torch.Size([16, 64])
Labels shape: torch.Size([16])


# Step 12 — Load Pretrained FinBERT

In [ ]:


from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("FinBERT model loaded successfully.")
print("Model:", MODEL_NAME)

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12171.09it/s]

FinBERT model loaded successfully.
Model: ProsusAI/finbert


# Step 13 — Device Configuration

In [ ]:


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Device: cpu
Running on CPU


# Step 14 — Baseline Sentiment Inference

In [ ]:


import torch.nn.functional as F

model.eval()

all_predictions = []
all_probabilities = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = F.softmax(outputs.logits, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Baseline inference completed.")
print("Number of predictions:", len(all_predictions))

Baseline inference completed.
Number of predictions: 340


# Step 15 — Baseline Evaluation

In [ ]:


from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

print("Baseline FinBERT Results")
print("------------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("Macro F1 :", round(f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            id2label[0],
            id2label[1],
            id2label[2]
        ],
        zero_division=0
    )
)

Baseline FinBERT Results
------------------------
Accuracy : 0.0088
Precision: 0.0177
Recall   : 0.0104
Macro F1 : 0.0102

Classification Report:
              precision    recall  f1-score   support

    negative       0.01      0.02      0.01        46
     neutral       0.04      0.01      0.02       209
    positive       0.00      0.00      0.00        85

    accuracy                           0.01       340
   macro avg       0.02      0.01      0.01       340
weighted avg       0.03      0.01      0.01       340



 FinBERT Embedding Extraction

The FinBERT [CLS] hidden representation is extracted as a 768-dimensional financial-text embedding for downstream QuantFormer multimodal fusion.

# Step 16 — Extract FinBERT Embeddings

In [ ]:


from transformers import AutoModel

# Load FinBERT encoder for hidden representations
embedding_model = AutoModel.from_pretrained(MODEL_NAME)
embedding_model = embedding_model.to(device)
embedding_model.eval()

def extract_embeddings(data_loader):
    embeddings = []
    labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = embedding_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # [CLS] token representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

            embeddings.append(cls_embeddings.cpu().numpy())
            labels.extend(batch["labels"].numpy())

    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.array(labels)

    return embeddings, labels


train_embeddings, train_embedding_labels = extract_embeddings(train_loader)
val_embeddings, val_embedding_labels = extract_embeddings(val_loader)
test_embeddings, test_embedding_labels = extract_embeddings(test_loader)

print("Embedding extraction completed.")
print("Train embeddings shape:", train_embeddings.shape)
print("Validation embeddings shape:", val_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6011.80it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding extraction completed.
Train embeddings shape: (1584, 768)
Validation embeddings shape: (340, 768)
Test embeddings shape: (340, 768)


# Step 17 — Validate Embeddings

In [ ]:


print("Train embedding labels match:",
      len(train_embeddings) == len(train_embedding_labels))

print("Validation embedding labels match:",
      len(val_embeddings) == len(val_embedding_labels))

print("Test embedding labels match:",
      len(test_embeddings) == len(test_embedding_labels))

print("\nEmbedding dimension:", train_embeddings.shape[1])

print("\nFirst embedding (first 10 values):")
print(train_embeddings[0][:10])

Train embedding labels match: True
Validation embedding labels match: True
Test embedding labels match: True

Embedding dimension: 768

First embedding (first 10 values):
[ 0.05841468  0.8454528  -0.2658209  -0.4427947   0.3108425  -0.71575844
  0.00376576  0.23163225  0.21285382  0.4939173 ]


 Save Embeddings and Labels

The generated FinBERT embeddings and corresponding labels are saved for downstream use.

# Step 18 — Save FinBERT Embeddings

In [ ]:


from pathlib import Path

OUTPUT_DIR = Path("../datasets/processed/phrasebank")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / "train_embeddings.npy", train_embeddings)
np.save(OUTPUT_DIR / "val_embeddings.npy", val_embeddings)
np.save(OUTPUT_DIR / "test_embeddings.npy", test_embeddings)

np.save(
    OUTPUT_DIR / "train_labels.npy",
    train_embedding_labels
)

np.save(
    OUTPUT_DIR / "val_labels.npy",
    val_embedding_labels
)

np.save(
    OUTPUT_DIR / "test_labels.npy",
    test_embedding_labels
)

print("Embeddings and labels saved successfully.")
print("Saved to:", OUTPUT_DIR)

Embeddings and labels saved successfully.
Saved to: ..\datasets\processed\phrasebank


# Step 19 — Save Label Mapping and Configuration

In [ ]:


import json

config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "embedding_dimension": int(train_embeddings.shape[1]),
    "seed": SEED,
    "label2id": label2id,
    "id2label": id2label
}

with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Configuration saved successfully.")
print(json.dumps(config, indent=4))

Configuration saved successfully.
{
    "model_name": "ProsusAI/finbert",
    "max_length": 64,
    "embedding_dimension": 768,
    "seed": 42,
    "label2id": {
        "negative": 0,
        "neutral": 1,
        "positive": 2
    },
    "id2label": {
        "0": "negative",
        "1": "neutral",
        "2": "positive"
    }
}


 Verify Saved Artifacts

The saved embeddings, labels, and configuration files are verified to ensure successful artifact generation.

# Step 20 — Verify Saved Artifacts

In [ ]:


print("Saved files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)

Saved files:
- label_mapping.json
- test_embeddings.npy
- test_labels.npy
- train.pt
- train_embeddings.npy
- train_labels.npy
- val.pt
- val_embeddings.npy
- val_labels.npy


 Final Pipeline Validation

The complete FinBERT pipeline is validated from dataset preparation through sentiment prediction and embedding extraction.

# Step 21 — Final Pipeline Validation

In [ ]:


print("===== Q4 FinBERT Pipeline Validation =====")

print("\n1. Dataset")
print("Total samples:", len(df))
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("\n2. Tokenization")
print("MAX_LENGTH:", MAX_LENGTH)
print("Train token shape:", np.array(train_encodings["input_ids"]).shape)

print("\n3. Baseline Sentiment")
print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(f1, 4))

print("\n4. Embeddings")
print("Train:", train_embeddings.shape)
print("Validation:", val_embeddings.shape)
print("Test:", test_embeddings.shape)

print("\n5. Saved Artifacts")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)

print("\n===== Q4 PIPELINE COMPLETED SUCCESSFULLY =====")

===== Q4 FinBERT Pipeline Validation =====

1. Dataset
Total samples: 2264
Train samples: 1584
Validation samples: 340
Test samples: 340

2. Tokenization
MAX_LENGTH: 64
Train token shape: (1584, 64)

3. Baseline Sentiment
Accuracy: 0.0088
Macro F1: 0.0102

4. Embeddings
Train: (1584, 768)
Validation: (340, 768)
Test: (340, 768)

5. Saved Artifacts
- label_mapping.json
- test_embeddings.npy
- test_labels.npy
- train.pt
- train_embeddings.npy
- train_labels.npy
- val.pt
- val_embeddings.npy
- val_labels.npy

===== Q4 PIPELINE COMPLETED SUCCESSFULLY =====


# 22. Final Notebook Summary

This notebook implemented a complete Financial PhraseBank to FinBERT sentiment and embedding pipeline.

The workflow included:

- Loading and validating the Financial PhraseBank dataset.
- Encoding negative, neutral, and positive sentiment labels.
- Creating stratified train, validation, and test splits.
- Tokenizing financial sentences using the FinBERT tokenizer.
- Creating PyTorch datasets and DataLoaders.
- Loading the pretrained ProsusAI/FinBERT model.
- Performing baseline financial sentiment prediction.
- Evaluating the model using accuracy, precision, recall, macro F1-score, and classification metrics.
- Extracting 768-dimensional FinBERT [CLS] embeddings.
- Saving embeddings, labels, and model configuration for downstream use.

The generated FinBERT embeddings will be used as financial-text representations for the future multimodal QuantFormer fusion stage.

No artificial timestamp-based pairing between Financial PhraseBank sentences and FI-2010 order-book observations was performed in this notebook.